In [ ]:
import numpy as np
import pandas as pd
import panel as pn
import holoviews as hv
from holoviews import opts
from holoviews.operation.datashader import rasterize, datashade
import datashader as ds
from datashader.colors import viridis, inferno
import colorcet as cc
import time
import sys
from io import StringIO
from contextlib import redirect_stdout, redirect_stderr

# Enable Panel extension for Jupyter
pn.extension()
hv.extension('bokeh')

# Import your simulation modules
from simulation import run_simulation
from config import SimRun


class InteractiveInterferogram:
    """
    Interactive interferogram visualization using Datashader + Panel.
    Optimized for Jupyter Lab and Google Colab.
    """
    
    def __init__(
        self,
        eps0_min,
        eps0_max,
        A_min,
        A_max,
        N_points_target,
        delta_C_range,
        GammaL0_range,
        GammaR0_range,
        Gamma_eg0_range,
        Gamma_phi0_range,
        N_steps_period_array,
        N_periods_array,
        N_periods_avg_array,
        delta_C_default,
        GammaL0_default,
        GammaR0_default,
        Gamma_eg0_default,
        Gamma_phi0_default,
        N_steps_period_default,
        N_periods_default,
        N_periods_avg_default,
        dC_default_thresholds,
        platform_type=platform_type,
        repo_path=repo_path,
        cmap_name='fire'
    ):
        """Initialize the interactive interferogram viewer."""

        # Store platform and path information
        self.platform_type = platform_type
        self.repo_path = repo_path
        
        # Store grid inputs
        self.eps0_min = eps0_min
        self.eps0_max = eps0_max
        self.A_min = A_min
        self.A_max = A_max
        self.N_points_target = N_points_target
        
        # Store parameter ranges
        self.delta_C_range = delta_C_range
        self.GammaL0_range = GammaL0_range
        self.GammaR0_range = GammaR0_range
        self.Gamma_eg0_range = Gamma_eg0_range
        self.Gamma_phi0_range = Gamma_phi0_range
        
        # Store discrete parameter ranges (can be tuples)
        if isinstance(N_steps_period_array, tuple):
            self.N_steps_period_range = N_steps_period_array
        else:
            self.N_steps_period_range = (int(N_steps_period_array[0]), int(N_steps_period_array[-1]))
            
        if isinstance(N_periods_array, tuple):
            self.N_periods_range = N_periods_array
        else:
            self.N_periods_range = (int(N_periods_array[0]), int(N_periods_array[-1]))
            
        if isinstance(N_periods_avg_array, tuple):
            self.N_periods_avg_range = N_periods_avg_array
        else:
            self.N_periods_avg_range = (int(N_periods_avg_array[0]), int(N_periods_avg_array[-1]))
        
        # Colormap and thresholds
        self.cmap_name = cmap_name
        self.dC_default_thresholds = dC_default_thresholds
        
        # Level labels
        self.level_labels = ["00", "01", "10", "11", "C", "dC"]
        
        # Initialize data storage
        self.avg_grids = {label: None for label in self.level_labels}
        self.current_data = None
        
        # Initialize current parameters
        self.delta_C = delta_C_default
        self.GammaL0 = GammaL0_default
        self.GammaR0 = GammaR0_default
        self.Gamma_eg0 = Gamma_eg0_default
        self.Gamma_phi0 = Gamma_phi0_default
        self.N_steps_period = N_steps_period_default
        self.N_periods = N_periods_default
        self.N_periods_avg = N_periods_avg_default
        
        # Auto-update state
        self.auto_update_enabled = False
        
        # Add a counter to force plot updates
        self.data_version = 0
        
        # Flag to prevent double execution
        self._is_generating = False
        
        # Create widgets
        self._create_widgets()
        
        # Generate initial data
        self._generate_data()
        
    def _create_widgets(self):
        """Create all Panel widgets."""
        
        # === Continuous parameter sliders with text inputs ===
        self.delta_C_slider = pn.widgets.FloatSlider(
            name='delta_C',
            start=self.delta_C_range[0],
            end=self.delta_C_range[1],
            value=self.delta_C,
            step=(self.delta_C_range[1] - self.delta_C_range[0]) / 1000,
            format='0.5e'
        )
        self.delta_C_input = pn.widgets.FloatInput(
            value=self.delta_C,
            format='0.5e',
            width=100
        )
        
        self.GammaL0_slider = pn.widgets.FloatSlider(
            name='GammaL0',
            start=self.GammaL0_range[0],
            end=self.GammaL0_range[1],
            value=self.GammaL0,
            step=0.5
        )
        self.GammaL0_input = pn.widgets.FloatInput(
            value=self.GammaL0,
            width=100
        )
        
        self.GammaR0_slider = pn.widgets.FloatSlider(
            name='GammaR0',
            start=self.GammaR0_range[0],
            end=self.GammaR0_range[1],
            value=self.GammaR0,
            step=0.1
        )
        self.GammaR0_input = pn.widgets.FloatInput(
            value=self.GammaR0,
            width=100
        )
        
        self.Gamma_eg0_slider = pn.widgets.FloatSlider(
            name='Gamma_eg0',
            start=self.Gamma_eg0_range[0],
            end=self.Gamma_eg0_range[1],
            value=self.Gamma_eg0,
            step=0.1
        )
        self.Gamma_eg0_input = pn.widgets.FloatInput(
            value=self.Gamma_eg0,
            width=100
        )
        
        self.Gamma_phi0_slider = pn.widgets.FloatSlider(
            name='Gamma_phi0',
            start=self.Gamma_phi0_range[0],
            end=self.Gamma_phi0_range[1],
            value=self.Gamma_phi0,
            step=0.1
        )
        self.Gamma_phi0_input = pn.widgets.FloatInput(
            value=self.Gamma_phi0,
            width=100
        )
        
        # === Discrete parameter sliders ===
        self.N_steps_period_slider = pn.widgets.IntSlider(
            name='N_steps_period',
            start=self.N_steps_period_range[0],
            end=self.N_steps_period_range[1],
            value=self.N_steps_period,
            step=1
        )
        self.N_steps_period_input = pn.widgets.IntInput(
            value=self.N_steps_period,
            width=100
        )
        
        self.N_periods_slider = pn.widgets.IntSlider(
            name='N_periods',
            start=self.N_periods_range[0],
            end=self.N_periods_range[1],
            value=self.N_periods,
            step=1
        )
        self.N_periods_input = pn.widgets.IntInput(
            value=self.N_periods,
            width=100
        )
        
        self.N_periods_avg_slider = pn.widgets.IntSlider(
            name='N_periods_avg',
            start=self.N_periods_avg_range[0],
            end=self.N_periods_avg_range[1],
            value=self.N_periods_avg,
            step=1
        )
        self.N_periods_avg_input = pn.widgets.IntInput(
            value=self.N_periods_avg,
            width=100
        )
        
        # Link sliders and inputs bidirectionally
        self._link_slider_input('delta_C')
        self._link_slider_input('GammaL0')
        self._link_slider_input('GammaR0')
        self._link_slider_input('Gamma_eg0')
        self._link_slider_input('Gamma_phi0')
        self._link_slider_input('N_steps_period')
        self._link_slider_input('N_periods')
        self._link_slider_input('N_periods_avg')
        
        # === Level selector ===
        self.level_selector = pn.widgets.RadioButtonGroup(
            name='Display Level',
            options=['00', '01', '10', '11', 'C', 'dC'],
            value='dC',
            button_type='primary'
        )
        
        # === Color range sliders ===
        self.clim_low_slider = pn.widgets.FloatSlider(
            name='Color Min',
            start=-10000,
            end=10000,
            value=self.dC_default_thresholds[0],
            step=10
        )
        self.clim_low_input = pn.widgets.FloatInput(
            value=self.dC_default_thresholds[0],
            width=100
        )
        
        self.clim_high_slider = pn.widgets.FloatSlider(
            name='Color Max',
            start=-10000,
            end=10000,
            value=self.dC_default_thresholds[1],
            step=10
        )
        self.clim_high_input = pn.widgets.FloatInput(
            value=self.dC_default_thresholds[1],
            width=100
        )
        
        # Link color sliders and inputs
        self._link_slider_input('clim_low')
        self._link_slider_input('clim_high')
        
        # === Update button ===
        self.update_button = pn.widgets.Button(
            name='🔄 Regenerate Data',
            button_type='success',
            width=180
        )
        
        # === Auto-update toggle ===
        self.auto_update_toggle = pn.widgets.Toggle(
            name='Auto Update',
            value=False,
            button_type='warning',
            width=120
        )
        
        # === Status indicator ===
        self.status_text = pn.pane.Markdown(
            "**Status:** Ready",
            width=300,
            sizing_mode='fixed'
        )
        
        # === Computation time display ===
        self.timing_text = pn.pane.Markdown(
            "**Last computation:** N/A",
            width=300,
            sizing_mode='fixed'
        )
        
        # === Log display area (scrollable) ===
        self.log_display = pn.pane.Markdown(
            "**Log:** Waiting for first computation...",
            width=650,
            height=150,
            styles={'overflow-y': 'scroll', 'background': '#f5f5f5', 'padding': '10px', 
                   'font-family': 'monospace', 'font-size': '11px', 'border': '1px solid #ddd'},
            sizing_mode='fixed'
        )
        
        # Watch level selector to update color limits
        self.level_selector.param.watch(self._on_level_change, 'value')
        
        # Watch auto-update toggle
        self.auto_update_toggle.param.watch(self._on_auto_update_toggle, 'value')
        
        # Watch simulation parameter sliders for auto-update (NOT color sliders)
        for slider_name in ['delta_C', 'GammaL0', 'GammaR0', 'Gamma_eg0', 'Gamma_phi0',
                           'N_steps_period', 'N_periods', 'N_periods_avg']:
            slider = getattr(self, f'{slider_name}_slider')
            slider.param.watch(self._on_slider_change, 'value')
    
    def _link_slider_input(self, param_name):
        """Link a slider and input box bidirectionally."""
        slider = getattr(self, f'{param_name}_slider')
        input_box = getattr(self, f'{param_name}_input')
        
        def slider_to_input(event):
            input_box.value = event.new
        
        def input_to_slider(event):
            if event.new is not None:
                # Clamp to slider range
                val = max(slider.start, min(slider.end, event.new))
                slider.value = val
                if val != event.new:
                    input_box.value = val
        
        slider.param.watch(slider_to_input, 'value')
        input_box.param.watch(input_to_slider, 'value')
        
    def _on_level_change(self, event):
        """Update color limits when level changes."""
        level = event.new
        data = self._compute_level_data(level)
        
        data_min, data_max = float(data.min()), float(data.max())
        
        # Update slider ranges to actual data range
        self.clim_low_slider.start = data_min
        self.clim_low_slider.end = data_max
        self.clim_high_slider.start = data_min
        self.clim_high_slider.end = data_max
        
        # Set appropriate default values based on level
        if level in ["00", "01", "10", "11"]:
            self.clim_low_slider.value = max(data_min, 0)
            self.clim_high_slider.value = min(data_max, 1)
        elif level == "C":
            self.clim_low_slider.value = max(data_min, -18)
            self.clim_high_slider.value = min(data_max, 18)
        elif level == "dC":
            self.clim_low_slider.value = max(data_min, 2*self.dC_default_thresholds[0])
            self.clim_high_slider.value = min(data_max, 2*self.dC_default_thresholds[1])
        
        # Update input boxes
        self.clim_low_input.value = self.clim_low_slider.value
        self.clim_high_input.value = self.clim_high_slider.value
    
    def _on_auto_update_toggle(self, event):
        """Handle auto-update toggle."""
        self.auto_update_enabled = event.new
        if self.auto_update_enabled:
            self.update_button.name = '🔄 Update (Auto On)'
            self.update_button.button_type = 'light'
        else:
            self.update_button.name = '🔄 Regenerate Data'
            self.update_button.button_type = 'success'
    
    def _on_slider_change(self, event):
        """Handle slider changes for auto-update."""
        if self.auto_update_enabled:
            self._update_params_and_regenerate()
    
    def _update_params_and_regenerate(self, event=None):
        """Update current parameters from sliders and regenerate."""
        
        # Update current parameters from sliders
        self.delta_C = self.delta_C_slider.value
        self.GammaL0 = self.GammaL0_slider.value
        self.GammaR0 = self.GammaR0_slider.value
        self.Gamma_eg0 = self.Gamma_eg0_slider.value
        self.Gamma_phi0 = self.Gamma_phi0_slider.value
        self.N_steps_period = self.N_steps_period_slider.value
        self.N_periods = self.N_periods_slider.value
        self.N_periods_avg = self.N_periods_avg_slider.value
        
        # Regenerate data
        self._generate_data()
    
    def _generate_data(self):
        """Generate interferogram data based on current parameters."""
        
        # Prevent double execution
        if self._is_generating:
            return
        
        self._is_generating = True
        
        # Capture stdout/stderr
        captured_stdout = StringIO()
        captured_stderr = StringIO()
        
        with redirect_stdout(captured_stdout), redirect_stderr(captured_stderr):
            self.status_text.object = "**Status:** 🔄 Generating data..."
            start_time = time.perf_counter()
            
            simr = SimRun(
                delta_C=self.delta_C,
                GammaL0=self.GammaL0,
                GammaR0=self.GammaR0,
                Gamma_eg0=self.Gamma_eg0,
                Gamma_phi0=self.Gamma_phi0,
                eps0_min=self.eps0_min,
                eps0_max=self.eps0_max,
                A_min=self.A_min,
                A_max=self.A_max,
                N_points_target=self.N_points_target,
                N_steps_period=self.N_steps_period,
                N_periods=self.N_periods,
                N_periods_avg=self.N_periods_avg,
                platform_type=self.platform_type,
                repo_path=self.repo_path
            )
            
            self.eps0_grid, self.A_grid, rho_avg_cdc_3d, _ = run_simulation(simr)
            
            # Store the averaged grids
            for i, label in enumerate(self.level_labels):
                self.avg_grids[label] = rho_avg_cdc_3d[i]
            
            end_time = time.perf_counter()
            elapsed = end_time - start_time
        
        self.status_text.object = "**Status:** ✅ Data ready"
        self.timing_text.object = f"**Last computation:** {elapsed:.2f} seconds"
        
        # Build log text
        log_text = f"**Computation completed in {elapsed:.2f}s**\n\n"
        log_text += f"**Parameters:**\n"
        log_text += f"- delta_C = {self.delta_C:.6e}\n"
        log_text += f"- GammaL0 = {self.GammaL0}, GammaR0 = {self.GammaR0}\n"
        log_text += f"- Gamma_eg0 = {self.Gamma_eg0}, Gamma_phi0 = {self.Gamma_phi0}\n"
        log_text += f"- N_steps = {self.N_steps_period}, N_periods = {self.N_periods}, N_avg = {self.N_periods_avg}\n\n"
        
        stdout_content = captured_stdout.getvalue()
        stderr_content = captured_stderr.getvalue()
        
        if stdout_content:
            log_text += f"**Output:**\n```\n{stdout_content}\n```\n"
        if stderr_content:
            log_text += f"**Warnings:**\n```\n{stderr_content}\n```\n"
        
        self.log_display.object = log_text
        
        # Increment data version to trigger plot update
        self.data_version += 1
        if hasattr(self, 'data_version_widget'):
            self.data_version_widget.value = self.data_version
        
        self._is_generating = False
        
    def _compute_level_data(self, level):
        """Return data for the selected level."""
        
        if level in ["00", "01", "10", "11", "C", "dC"]:
            return self.avg_grids[level]
        else:
            return self.avg_grids["00"]
    
    def view_plot(self, level=None, clim_low=None, clim_high=None, data_version=None):
        """Create the HoloViews plot with datashader."""
        
        if level is None:
            level = self.level_selector.value
        if clim_low is None:
            clim_low = self.clim_low_slider.value
        if clim_high is None:
            clim_high = self.clim_high_slider.value
        
        # data_version parameter triggers updates when data changes
        data = self._compute_level_data(level)
        
        img = hv.Image(
            (self.eps0_grid, self.A_grid, data),
            kdims=['eps0', 'A'],
            vdims=['value']
        )
        
        img = img.opts(
            cmap=self.cmap_name,
            colorbar=True,
            width=800,
            height=600,
            clim=(clim_low, clim_high),
            title=f'Interactive Interferogram - Level: {level}',
            xlabel='eps0',
            ylabel='A',
            tools=['hover', 'box_zoom', 'wheel_zoom', 'pan', 'reset'],
            active_tools=['wheel_zoom']
        )
        
        return img

    def create_dashboard(self):
        """Create the complete Panel dashboard."""
        
        # Connect update button
        self.update_button.on_click(self._update_params_and_regenerate)
        
        # Hidden widget to trigger plot updates
        self.data_version_widget = pn.widgets.IntInput(value=0, visible=False)
        
        # Create a bound plot function
        @pn.depends(
            self.level_selector.param.value,
            self.clim_low_slider.param.value,
            self.clim_high_slider.param.value,
            self.data_version_widget.param.value,
            watch=False
        )
        def plot_view(level, clim_low, clim_high, data_version):
            return self.view_plot(level, clim_low, clim_high, data_version)
        
        # Create layout
        sidebar = pn.Column(
            "## 🎛️ Simulation Parameters",
            pn.layout.Divider(),
            "### Physical Parameters",
            pn.Row(self.delta_C_slider, self.delta_C_input),
            pn.Row(self.GammaL0_slider, self.GammaL0_input),
            pn.Row(self.GammaR0_slider, self.GammaR0_input),
            pn.Row(self.Gamma_eg0_slider, self.Gamma_eg0_input),
            pn.Row(self.Gamma_phi0_slider, self.Gamma_phi0_input),
            pn.layout.Divider(),
            "### Time Parameters",
            pn.Row(self.N_steps_period_slider, self.N_steps_period_input),
            pn.Row(self.N_periods_slider, self.N_periods_input),
            pn.Row(self.N_periods_avg_slider, self.N_periods_avg_input),
            pn.layout.Divider(),
            pn.Row(self.update_button, self.auto_update_toggle),
            self.status_text,
            self.timing_text,
            width=500,
            sizing_mode='fixed'
        )
        
        plot_and_controls = pn.Column(
            pn.Row(
                pn.Column("**Level:**", self.level_selector),
                pn.Column(
                    pn.Row(self.clim_low_slider, self.clim_low_input),
                    pn.Row(self.clim_high_slider, self.clim_high_input)
                )
            ),
            plot_view,
            pn.layout.Divider(),
            "### Computation Log",
            self.log_display,
            sizing_mode='fixed'
        )
        
        dashboard = pn.Row(
            sidebar,
            plot_and_controls,
            sizing_mode='fixed'
        )
        
        return dashboard

Cloning the repository...
Repository cloned successfully.
Platform: local_windows
Repo directory: C:\Users\E-Store\cuda_python_project
Parent directory: C:\Users\E-Store
Changing directory to: C:\Users\E-Store


In [4]:
import subprocess
import os

# Change directory to the repo path
os.chdir(repo_path)

# Run git pull and capture output
try:
    result = subprocess.run(['git', 'pull', 'origin', 'main'], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    # Check if the output indicates that changes were pulled
    if "Already up to date." in result.stdout:
        print("Already up to date.")
    else:
        print(result.stdout)  # Print the pull output (new commits, files updated, etc.)
        print("Changes were pulled from the repository.")
except subprocess.CalledProcessError as e:
    print(f"Error pulling changes: {e.stderr}")


Already up to date.


In [6]:
# Full paths to the build/setup scripts
setup_colab_script = repo_path / 'scripts' / 'setup_colab.sh'
linux_build_script = repo_path / 'scripts' / 'build_local_linux.sh'
windows_build_script = repo_path / 'scripts' / 'build_local_windows.bat'

# Password for sudo (for WSL2 or Local Linux)
sudo_password = '7566456'  # Replace with your actual password

try:
    if platform_type in ['local_wsl2', 'local_linux']:
        # For WSL2 or Local Linux, use sudo to run the build script
        print("Building for Linux/WSL2...")
        result = subprocess.run(
            f'echo {sudo_password} | sudo -S bash {linux_build_script}',
            shell=True,
            check=True,
            capture_output=True,
            text=True
        )
        print(result.stdout)
        if result.stderr:
            print("Warnings/Errors:", result.stderr)
        print("Build script executed for local Linux/WSL2 environment.")
    
    elif platform_type == 'colab':
        # For Colab, just run the setup script without sudo
        print("Building for Google Colab...")
        result = subprocess.run(
            f'bash {setup_colab_script}',
            shell=True,
            check=True,
            capture_output=True,
            text=True
        )
        print(result.stdout)
        if result.stderr:
            print("Warnings/Errors:", result.stderr)
        print("Setup script executed in Colab.")
        
    elif platform_type == 'local_windows':
        # For Windows, run the batch script
        print("Building for Windows...")
        if not windows_build_script.exists():
            print(f"Error: Build script not found at {windows_build_script}")
            sys.exit(1)
            
        # Run the batch script from its directory to ensure relative paths work
        result = subprocess.run(
            [str(windows_build_script)],
            cwd=windows_build_script.parent,
            check=True,
            capture_output=True,
            text=True,
            shell=True  # Needed for batch files on Windows
        )
        print(result.stdout)
        if result.stderr:
            print("Warnings/Errors:", result.stderr)
        print("Build script executed for Windows environment.")
        
    else:
        print(f"Error: Unknown platform_type '{platform_type}'")
        sys.exit(1)

except FileNotFoundError as e:
    print(f"Error: Required file or command not found: {e}")
    sys.exit(1)
    
except subprocess.CalledProcessError as e:
    print(f"Error: Build script failed with exit code {e.returncode}")
    print(f"stdout: {e.stdout}")
    print(f"stderr: {e.stderr}")
    sys.exit(1)
    
except Exception as e:
    print(f"Unexpected error: {e}")
    sys.exit(1)

print("\n✓ Build completed successfully!")

Building for Windows...
Building CUDA code on Windows...
Detected GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
Using CUDA architecture: sm_86
Found NVCC at: C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.1\bin\nvcc.exe
**********************************************************************
** Visual Studio 2022 Developer Command Prompt v17.14.13
** Copyright (c) 2025 Microsoft Corporation
**********************************************************************
[vcvarsall.bat] Environment initialized for: 'x64'
Compiling CUDA code...
C:\Users\E-Store\cuda_python_project\cuda\src\host/host_branch_grid.cuh(78): warning #550-D: variable "rho_avg_dim_u" was set but never used
      int rho_avg_dim, rho_avg_dim_u;
                       ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

main.cu
tmpxft_00004eb0_00000000-10_main.cudafe1.cpp
C:\Users\E-Store\cuda_python_project\cuda\src\host\host_branch_grid.cuh(285) : warning C4700: uninitialized local variabl

In [9]:
# Ensure repo_path is a string for subprocess
repo_path_str = str(repo_path)

# Determine the command to run based on the platform
if platform_type in ['colab_linux', 'local_linux', 'local_wsl2']:
    bin_command = f"./cuda/bin/lindblad_gpu grid last bin_file unrolled MySimSharedMemory -0.006 0.006 0.0 0.01 774 645 1000 10 1 70819 21 0.0002 0.0033 0.25 0.25 0.25 0.25 {repo_path_str}/cuda/output/rho_avg_out.csv {repo_path_str}/cuda/output/rho_avg_out.bin {repo_path_str}/cuda/output/rho_dynamics_single_mode_out.csv 0.00011608757555650906 0 0 0.8 3.6 50.0 225.0 12.0 54.0 50.0 12.0 0 0 0 0.8 0.0015731484686413405 3.6 False {repo_path_str}/cuda/output/rho_dynamics_single_mode_log_out.csv {repo_path_str}/cuda/output/rho_dynamics_single_mode_log_out.h5 one_thread_per_traj"
elif platform_type == 'local_windows':
    bin_command = f"{repo_path_str}\\cuda\\bin\\lindblad_gpu.exe grid last bin_file unrolled MySimSharedMemory -0.006 0.006 0.0 0.01 774 645 1000 10 1 70819 21 0.0002 0.0033 0.25 0.25 0.25 0.25 {repo_path_str}\\cuda\\output\\rho_avg_out.csv {repo_path_str}\\cuda\\output\\rho_avg_out.bin {repo_path_str}\\cuda\\output\\rho_dynamics_single_mode_out.csv 0.00011608757555650906 0 0 0.8 3.6 50.0 225.0 12.0 54.0 50.0 12.0 0 0 0 0.8 0.0015731484686413405 3.6 False {repo_path_str}\\cuda\\output\\rho_dynamics_single_mode_log_out.csv {repo_path_str}\\cuda\\output\\rho_dynamics_single_mode_log_out.h5 one_thread_per_traj"

# Change directory to the repo path (before running the command)
os.chdir(repo_path)

# Print the command for debugging
print(f"Running command: {bin_command}")

# If we're on Windows, use cmd /c to execute the command in a shell
if platform_type == 'local_windows':
    bin_command = f"cmd /c \"{bin_command}\""

# Run the command and capture output
try:
    result = subprocess.run(bin_command, shell=True, check=True, capture_output=True, text=True)
    print("\nCommand stdout:", result.stdout)  # Print standard output
    print("Command stderr:", result.stderr)  # Print standard error (if any)
except subprocess.CalledProcessError as e:
    print(f"Error occurred while running the command: {e}")
    print(f"Error output: {e.stderr}")


Running command: C:\Users\E-Store\cuda_python_project\cuda\bin\lindblad_gpu.exe grid last bin_file unrolled MySimSharedMemory -0.006 0.006 0.0 0.01 774 645 1000 10 1 70819 21 0.0002 0.0033 0.25 0.25 0.25 0.25 C:\Users\E-Store\cuda_python_project\cuda\output\rho_avg_out.csv C:\Users\E-Store\cuda_python_project\cuda\output\rho_avg_out.bin C:\Users\E-Store\cuda_python_project\cuda\output\rho_dynamics_single_mode_out.csv 0.00011608757555650906 0 0 0.8 3.6 50.0 225.0 12.0 54.0 50.0 12.0 0 0 0 0.8 0.0015731484686413405 3.6 False C:\Users\E-Store\cuda_python_project\cuda\output\rho_dynamics_single_mode_log_out.csv C:\Users\E-Store\cuda_python_project\cuda\output\rho_dynamics_single_mode_log_out.h5 one_thread_per_traj

Command stdout: 
--- Listing received arguments ---
1.  single_mode_flag: grid
2.  avg_periods_ouput_option: last
3.  ouput_option: bin_file
4.  unrolled_option: unrolled
5.  ram_shared_mmap_name: MySimSharedMemory
6.  eps0_min: -0.006
7.  eps0_max: 0.006
8.  A_min: 0.0
9.  A_ma

In [10]:
# Determine the command to run based on platform
script_path = repo_path / 'python' / 'single_interferogram.py'  # Using pathlib to join paths
python_command = "python3" if platform_type in ['colab_linux', 'local_linux', 'local_wsl2'] else "python"

# Construct the command to run the Python script
command = [python_command, str(script_path)]  # Command as a list to prevent issues with spaces

# Print the command for debugging
print(f"Running command: {' '.join(command)}")

# Run the command using subprocess
try:
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    print("Command stdout:", result.stdout)  # Print standard output
    print("Command stderr:", result.stderr)  # Print standard error (if any)
except subprocess.CalledProcessError as e:
    print(f"Error occurred while running the command: {e}")
    print(f"Error output: {e.stderr}")

Running command: python C:\Users\E-Store\cuda_python_project\python\single_interferogram.py
Command stdout: Environment detected: Windows
cuda_program_path =  C:\Users\E-Store\Documents\projects\repos\lindblad_cuda3\x64\Release\lindblad_cuda3.exe
output_dir =  C:\Users\E-Store\Documents\projects\repos\lindblad_cuda3\x64\Release\output

Args being passed to CUDA program: ['C:\\Users\\E-Store\\Documents\\projects\\repos\\lindblad_cuda3\\x64\\Release\\lindblad_cuda3.exe', 'grid', 'last', 'bin_file', 'unrolled', 'MySimSharedMemory', '-0.006', '0.006', '0.0', '0.01', '1096', '913', '1000', '10', '1', '70819', '21', '0.0002', '0.0033', '0.25', '0.25', '0.25', '0.25', 'C:\\Users\\E-Store\\Documents\\projects\\repos\\lindblad_cuda3\\x64\\Release\\output\\rho_avg_out.csv', 'C:\\Users\\E-Store\\Documents\\projects\\repos\\lindblad_cuda3\\x64\\Release\\output\\rho_avg_out.bin', 'C:\\Users\\E-Store\\Documents\\projects\\repos\\lindblad_cuda3\\x64\\Release\\output\\rho_dynamics_single_mode_out.csv'

In [13]:

# Set the Python folder path (assuming it's in 'python' folder inside the repo)
python_folder = repo_path / 'python'

# Add the 'python' folder to sys.path to allow importing Python modules
sys.path.append(str(python_folder))  # Make sure 'python' folder is accessible for imports

# Change the current working directory to repo_path
os.chdir(str(python_folder))

In [14]:
sys.path

['C:\\Users\\E-Store',
 'C:\\Users\\E-Store\\anaconda3\\envs\\qutip-env-2\\python312.zip',
 'C:\\Users\\E-Store\\anaconda3\\envs\\qutip-env-2\\DLLs',
 'C:\\Users\\E-Store\\anaconda3\\envs\\qutip-env-2\\Lib',
 'C:\\Users\\E-Store\\anaconda3\\envs\\qutip-env-2',
 '',
 'C:\\Users\\E-Store\\anaconda3\\envs\\qutip-env-2\\Lib\\site-packages',
 'C:\\Users\\E-Store\\anaconda3\\envs\\qutip-env-2\\Lib\\site-packages\\win32',
 'C:\\Users\\E-Store\\anaconda3\\envs\\qutip-env-2\\Lib\\site-packages\\win32\\lib',
 'C:\\Users\\E-Store\\anaconda3\\envs\\qutip-env-2\\Lib\\site-packages\\Pythonwin',
 'C:\\Users\\E-Store\\anaconda3\\envs\\qutip-env-2\\Lib\\site-packages\\setuptools\\_vendor',
 'C:\\Users\\E-Store\\cuda_python_project\\python',
 'C:\\Users\\E-Store\\cuda_python_project\\python']

In [25]:
import numpy as np
import pandas as pd
import panel as pn
import holoviews as hv
from holoviews import opts
from holoviews.operation.datashader import rasterize, datashade
import datashader as ds
from datashader.colors import viridis, inferno
import colorcet as cc
import time

# Enable Panel extension for Jupyter
pn.extension()
hv.extension('bokeh')




# Import your simulation modules
from simulation import run_simulation
from config import SimRun


class InteractiveInterferogram:
    """
    Interactive interferogram visualization using Datashader + Panel.
    Optimized for Jupyter Lab and Google Colab.
    """
    
    def __init__(
        self,
        eps0_min,
        eps0_max,
        A_min,
        A_max,
        N_points_target,
        delta_C_range,
        GammaL0_range,
        GammaR0_range,
        Gamma_eg0_range,
        Gamma_phi0_range,
        N_steps_period_array,
        N_periods_array,
        N_periods_avg_array,
        delta_C_default,
        GammaL0_default,
        GammaR0_default,
        Gamma_eg0_default,
        Gamma_phi0_default,
        N_steps_period_default,
        N_periods_default,
        N_periods_avg_default,
        dC_default_thresholds,
        cmap_name='fire'
    ):
        """Initialize the interactive interferogram viewer."""
        
        # Store grid inputs
        self.eps0_min = eps0_min
        self.eps0_max = eps0_max
        self.A_min = A_min
        self.A_max = A_max
        self.N_points_target = N_points_target
        
        # Store parameter ranges
        self.delta_C_range = delta_C_range
        self.GammaL0_range = GammaL0_range
        self.GammaR0_range = GammaR0_range
        self.Gamma_eg0_range = Gamma_eg0_range
        self.Gamma_phi0_range = Gamma_phi0_range
        
        # Store discrete parameter ranges (can be tuples)
        if isinstance(N_steps_period_array, tuple):
            self.N_steps_period_range = N_steps_period_array
        else:
            self.N_steps_period_range = (int(N_steps_period_array[0]), int(N_steps_period_array[-1]))
            
        if isinstance(N_periods_array, tuple):
            self.N_periods_range = N_periods_array
        else:
            self.N_periods_range = (int(N_periods_array[0]), int(N_periods_array[-1]))
            
        if isinstance(N_periods_avg_array, tuple):
            self.N_periods_avg_range = N_periods_avg_array
        else:
            self.N_periods_avg_range = (int(N_periods_avg_array[0]), int(N_periods_avg_array[-1]))
        
        # Colormap and thresholds
        self.cmap_name = cmap_name
        self.dC_default_thresholds = dC_default_thresholds
        
        # Level labels
        self.level_labels = ["00", "01", "10", "11", "C", "dC"]
        
        # Initialize data storage
        self.avg_grids = {label: None for label in self.level_labels}
        self.current_data = None
        
        # Initialize current parameters
        
        self.delta_C        = delta_C_default
        self.GammaL0        = GammaL0_default
        self.GammaR0        = GammaR0_default
        self.Gamma_eg0      = Gamma_eg0_default
        self.Gamma_phi0     = Gamma_phi0_default
        self.N_steps_period = N_steps_period_default
        self.N_periods      = N_periods_default
        self.N_periods_avg  = N_periods_avg_default
        
        # Auto-update state
        self.auto_update_enabled = False
        
        # Add a counter to force plot updates
        self.data_version = 0
        
        # Flag to prevent double execution
        self._is_generating = False
        
        # Create widgets
        self._create_widgets()
        
        # Generate initial data
        self._generate_data()
        
    def _create_widgets(self):
        """Create all Panel widgets."""
        
        # === Continuous parameter sliders with text inputs ===
        self.delta_C_slider = pn.widgets.FloatSlider(
            name='delta_C',
            start=self.delta_C_range[0],
            end=self.delta_C_range[1],
            value=self.delta_C,
            step=(self.delta_C_range[1] - self.delta_C_range[0]) / 1000,
            format='0.5e'
        )
        self.delta_C_input = pn.widgets.FloatInput(
            value=self.delta_C,
            format='0.5e',
            width=100
        )
        
        self.GammaL0_slider = pn.widgets.FloatSlider(
            name='GammaL0',
            start=self.GammaL0_range[0],
            end=self.GammaL0_range[1],
            value=self.GammaL0,
            step=0.5
        )
        self.GammaL0_input = pn.widgets.FloatInput(
            value=self.GammaL0,
            width=100
        )
        
        self.GammaR0_slider = pn.widgets.FloatSlider(
            name='GammaR0',
            start=self.GammaR0_range[0],
            end=self.GammaR0_range[1],
            value=self.GammaR0,
            step=0.1
        )
        self.GammaR0_input = pn.widgets.FloatInput(
            value=self.GammaR0,
            width=100
        )
        
        self.Gamma_eg0_slider = pn.widgets.FloatSlider(
            name='Gamma_eg0',
            start=self.Gamma_eg0_range[0],
            end=self.Gamma_eg0_range[1],
            value=self.Gamma_eg0,
            step=0.1
        )
        self.Gamma_eg0_input = pn.widgets.FloatInput(
            value=self.Gamma_eg0,
            width=100
        )
        
        self.Gamma_phi0_slider = pn.widgets.FloatSlider(
            name='Gamma_phi0',
            start=self.Gamma_phi0_range[0],
            end=self.Gamma_phi0_range[1],
            value=self.Gamma_phi0,
            step=0.1
        )
        self.Gamma_phi0_input = pn.widgets.FloatInput(
            value=self.Gamma_phi0,
            width=100,
            height=35
        )
        
        # === Discrete parameter sliders (compact) ===
        self.N_steps_period_slider = pn.widgets.IntSlider(
            name='N_steps_period',
            start=self.N_steps_period_range[0],
            end=self.N_steps_period_range[1],
            value=self.N_steps_period,
            step=1,
            sizing_mode='stretch_width',
            height=35
        )
        
        self.N_steps_period_input = pn.widgets.IntInput(
            value=self.N_steps_period,
            width=100,
            height=35
        )
        
        self.N_periods_slider = pn.widgets.IntSlider(
            name='N_periods',
            start=self.N_periods_range[0],
            end=self.N_periods_range[1],
            value=self.N_periods,
            step=1,
            sizing_mode='stretch_width',
            height=35
        )
        
        self.N_periods_input = pn.widgets.IntInput(
            value=self.N_periods,
            width=100,
            height=35
        )
        
        self.N_periods_avg_slider = pn.widgets.IntSlider(
            name='N_periods_avg',
            start=self.N_periods_avg_range[0],
            end=self.N_periods_avg_range[1],
            value=self.N_periods_avg,
            step=1,
            sizing_mode='stretch_width',
            height=35
        )
        
        self.N_periods_avg_input = pn.widgets.IntInput(
            value=self.N_periods_avg,
            width=100,
            height=35
        )
        
        # Link sliders and inputs bidirectionally
        self._link_slider_input('delta_C')
        self._link_slider_input('GammaL0')
        self._link_slider_input('GammaR0')
        self._link_slider_input('Gamma_eg0')
        self._link_slider_input('Gamma_phi0')
        self._link_slider_input('N_steps_period')
        self._link_slider_input('N_periods')
        self._link_slider_input('N_periods_avg')
        
        # === Level selector ===
        self.level_selector = pn.widgets.RadioButtonGroup(
            name='Display Level',
            options=['00', '01', '10', '11', 'C', 'dC'],
            value='dC',
            button_type='primary'
        )
        
        # === Color range sliders (will be auto-updated based on data) ===
        self.clim_low_slider = pn.widgets.FloatSlider(
            name='Color Min',
            start=-10000,
            end=10000,
            value=self.dC_default_thresholds[0],
            step=10
        )
        self.clim_low_input = pn.widgets.FloatInput(
            value=self.dC_default_thresholds[0],
            width=100
        )
        
        self.clim_high_slider = pn.widgets.FloatSlider(
            name='Color Max',
            start=-10000,
            end=10000,
            value=self.dC_default_thresholds[1],
            step=10
        )
        self.clim_high_input = pn.widgets.FloatInput(
            value=self.dC_default_thresholds[1],
            width=100
        )
        
        # Link color sliders and inputs
        self._link_slider_input('clim_low')
        self._link_slider_input('clim_high')
        
        # === Update button ===
        self.update_button = pn.widgets.Button(
            name='🔄 Regenerate Data',
            button_type='success',
            width=180
        )
        
        # === Auto-update toggle ===
        self.auto_update_toggle = pn.widgets.Toggle(
            name='Auto Update',
            value=False,
            button_type='warning',
            width=120
        )
        
        # === Status indicator ===
        self.status_text = pn.pane.Markdown(
            "**Status:** Ready",
            width=300,
            sizing_mode='fixed'
        )
        
        # === Computation time display ===
        self.timing_text = pn.pane.Markdown(
            "**Last computation:** N/A",
            width=300,
            sizing_mode='fixed'
        )
        
        # Watch level selector to update color limits
        self.level_selector.param.watch(self._on_level_change, 'value')
        
        # Watch auto-update toggle
        self.auto_update_toggle.param.watch(self._on_auto_update_toggle, 'value')
        
        # Watch all sliders for auto-update
        for slider_name in ['delta_C', 'GammaL0', 'GammaR0', 'Gamma_eg0', 'Gamma_phi0',
                           'N_steps_period', 'N_periods', 'N_periods_avg']:
            slider = getattr(self, f'{slider_name}_slider')
            slider.param.watch(self._on_slider_change, 'value')
    
    def _link_slider_input(self, param_name):
        """Link a slider and input box bidirectionally."""
        slider = getattr(self, f'{param_name}_slider')
        input_box = getattr(self, f'{param_name}_input')
        
        def slider_to_input(event):
            input_box.value = event.new
        
        def input_to_slider(event):
            if event.new is not None:
                # Clamp to slider range
                val = max(slider.start, min(slider.end, event.new))
                slider.value = val
                if val != event.new:
                    input_box.value = val
        
        slider.param.watch(slider_to_input, 'value')
        input_box.param.watch(input_to_slider, 'value')
        
    def _on_level_change(self, event):
        """Update color limits when level changes."""
        level = event.new
        data = self._compute_level_data(level)
        
        data_min, data_max = float(data.min()), float(data.max())
        
        # Update slider ranges to actual data range
        self.clim_low_slider.start = data_min
        self.clim_low_slider.end = data_max
        self.clim_high_slider.start = data_min
        self.clim_high_slider.end = data_max
        
        # Set appropriate default values based on level
        # These are starting points, but user can adjust to full range
        if level in ["00", "01", "10", "11"]:
            # Population levels: typically 0 to 1
            self.clim_low_slider.value = max(data_min, 0)
            self.clim_high_slider.value = min(data_max, 1)
        elif level == "C":
            # C values: typically -18 to +18
            self.clim_low_slider.value = max(data_min, -18)
            self.clim_high_slider.value = min(data_max, 18)
        elif level == "dC":
            # dC: use full data range but clamp to reasonable defaults
            self.clim_low_slider.value = max(data_min, 2*self.dC_default_thresholds[0])
            self.clim_high_slider.value = min(data_max, 2*self.dC_default_thresholds[1])
        
        # Update input boxes too
        self.clim_low_input.value = self.clim_low_slider.value
        self.clim_high_input.value = self.clim_high_slider.value
    
    def _on_auto_update_toggle(self, event):
        """Handle auto-update toggle."""
        self.auto_update_enabled = event.new
        if self.auto_update_enabled:
            self.update_button.name = '🔄 Update (Auto On)'
            self.update_button.button_type = 'light'
        else:
            self.update_button.name = '🔄 Regenerate Data'
            self.update_button.button_type = 'success'
    
    def _on_slider_change(self, event):
        """Handle slider changes for auto-update."""
        if self.auto_update_enabled:
            self._update_params_and_regenerate()
    
    def _update_params_and_regenerate(self):
        """Update current parameters from sliders and regenerate."""
        
        # Update current parameters from sliders
        self.delta_C = self.delta_C_slider.value
        self.GammaL0 = self.GammaL0_slider.value
        self.GammaR0 = self.GammaR0_slider.value
        self.Gamma_eg0 = self.Gamma_eg0_slider.value
        self.Gamma_phi0 = self.Gamma_phi0_slider.value
        self.N_steps_period = self.N_steps_period_slider.value
        self.N_periods = self.N_periods_slider.value
        self.N_periods_avg = self.N_periods_avg_slider.value
        
        # Regenerate data
        self._generate_data()
    
    def _generate_data(self):
        """Generate interferogram data based on current parameters."""
        
        self.status_text.object = "**Status:** 🔄 Generating data..."
        start_time = time.perf_counter()
        
        simr = SimRun(
            delta_C=self.delta_C,
            GammaL0=self.GammaL0,
            GammaR0=self.GammaR0,
            Gamma_eg0=self.Gamma_eg0,
            Gamma_phi0=self.Gamma_phi0,
            eps0_min=self.eps0_min,
            eps0_max=self.eps0_max,
            A_min=self.A_min,
            A_max=self.A_max,
            N_points_target=self.N_points_target,
            N_steps_period=self.N_steps_period,
            N_periods=self.N_periods,
            N_periods_avg=self.N_periods_avg
        )
        
        
        self.eps0_grid, self.A_grid, rho_avg_cdc_3d, _ = run_simulation(simr)
        
        
        # Store the averaged grids
        for i, label in enumerate(self.level_labels):
            self.avg_grids[label] = rho_avg_cdc_3d[i]
        
        end_time = time.perf_counter()
        elapsed = end_time - start_time
        
        self.status_text.object = "**Status:** ✅ Data ready"
        self.timing_text.object = f"**Last computation:** {elapsed:.2f} seconds"
        
    def _compute_level_data(self, level):
        """Return data for the selected level."""
        
        if level in ["00", "01", "10", "11", "C", "dC"]:
            return self.avg_grids[level]
        
    
    def view_plot(self):
        """Create the HoloViews plot with datashader."""
        
        level = self.level_selector.value
        
        # Create HoloViews Image
        # eps0_grid and A_grid are 1D arrays
        # data is 2D: (len(A_grid), len(eps0_grid))
        img = hv.Image(
            (self.eps0_grid, self.A_grid, self._compute_level_data(level)),
            kdims=['eps0', 'A'],
            vdims=['value']
        )
        
        # Apply options
        img = img.opts(
            cmap=self.cmap_name,
            colorbar=True,
            width=800,
            height=600,
            clim=(self.clim_low_slider.value, self.clim_high_slider.value),
            title=f'Interactive Interferogram - Level: {level}',
            xlabel='eps0',
            ylabel='A',
            tools=['hover', 'box_zoom', 'wheel_zoom', 'pan', 'reset'],
            active_tools=['wheel_zoom']
        )
        
        return img

    def create_dashboard(self):
        """Create the complete Panel dashboard."""
        
        # Connect update button
        self.update_button.on_click(self._update_params_and_regenerate)
        
        # Create a bound plot function that accepts the parameter values
        @pn.depends(
            self.level_selector.param.value,
            self.clim_low_slider.param.value,
            self.clim_high_slider.param.value,
            watch=False
        )
        def plot_view(level, clim_low_slider, clim_high_slider):
            return self.view_plot()
        
        # Create layout
        sidebar = pn.Column(
            "## 🎛️ Simulation Parameters",
            pn.layout.Divider(),
            "### Physical Parameters",
            self.delta_C_slider,
            self.GammaL0_slider,
            self.GammaR0_slider,
            self.Gamma_eg0_slider,
            self.Gamma_phi0_slider,
            pn.layout.Divider(),
            "### Time Parameters",
            self.N_steps_period_slider,
            self.N_periods_slider,
            self.N_periods_avg_slider,
            pn.layout.Divider(),
            self.update_button,
            self.status_text,
            self.timing_text,
            width=350
        )
        
        plot_controls = pn.Column(
            pn.Row(
                pn.Column("**Level:**", self.level_selector),
                pn.Column(self.clim_low_slider, self.clim_high_slider)
            ),
            plot_view
        )
        
        dashboard = pn.Row(
            sidebar,
            plot_controls
        )
        
        return dashboard


In [26]:
app = InteractiveInterferogram(
    eps0_min=-0.006,
    eps0_max=0.006,
    A_min=0,
    A_max=0.01,
    N_points_target=500_000,
    delta_C_range=(0, 0.0006),
    GammaL0_range=(0, 100),
    GammaR0_range=(0, 24),
    Gamma_eg0_range=(0, 16),
    Gamma_phi0_range=(0, 72),
    N_steps_period_array=(100, 2000),
    N_periods_array=(1, 20),
    N_periods_avg_array=(1, 10),
    delta_C_default=0.00011608757555650906,
    GammaL0_default=50,
    GammaR0_default=12,
    Gamma_eg0_default=0.8,
    Gamma_phi0_default=3.6,
    N_steps_period_default=1000,
    N_periods_default=10,
    N_periods_avg_default=1,
    dC_default_thresholds=(-2000, 1000),
    cmap_name='fire'
)
dashboard = app.create_dashboard()

Environment detected: Windows
cuda_program_path =  C:\Users\E-Store\Documents\projects\repos\lindblad_cuda3\x64\Release\lindblad_cuda3.exe
output_dir =  C:\Users\E-Store\Documents\projects\repos\lindblad_cuda3\x64\Release\output

Args being passed to CUDA program: ['C:\\Users\\E-Store\\Documents\\projects\\repos\\lindblad_cuda3\\x64\\Release\\lindblad_cuda3.exe', 'grid', 'last', 'bin_file', 'unrolled', 'MySimSharedMemory', '-0.006', '0.006', '0.0', '0.01', '774', '645', '1000', '10', '1', '70819', '21', '0.0002', '0.0033', '0.25', '0.25', '0.25', '0.25', 'C:\\Users\\E-Store\\Documents\\projects\\repos\\lindblad_cuda3\\x64\\Release\\output\\rho_avg_out.csv', 'C:\\Users\\E-Store\\Documents\\projects\\repos\\lindblad_cuda3\\x64\\Release\\output\\rho_avg_out.bin', 'C:\\Users\\E-Store\\Documents\\projects\\repos\\lindblad_cuda3\\x64\\Release\\output\\rho_dynamics_single_mode_out.csv', '0.00011608757555650906', '0', '0', '0.8', '3.6', '50.0', '225.0', '12.0', '54.0', '50.0', '12.0', '0', '0'

In [27]:
dashboard

Traceback (most recent call last):
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\pyviz_comms\__init__.py", line 341, in _handle_msg
 self._on_msg(msg)
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\panel\viewable.py", line 494, in _on_msg
 patch.apply_to_document(doc, comm.id if comm else None)
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\bokeh\protocol\messages\patch_doc.py", line 104, in apply_to_document
 invoke_with_curdoc(doc, lambda: doc.apply_json_patch(self.payload, setter=setter))
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\bokeh\document\callbacks.py", line 453, in invoke_with_curdoc
 return f()
 ^^^
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\bokeh\protocol\messages\patch_doc.py", line 104, in <lambda>
 invoke_with_curdoc(doc, lambda: doc.apply_json_patch(self.payload, setter=setter))
 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\bokeh\document\document.py", line 391, in apply_json_patch
 DocumentPatchedEvent.handle_event(self, event, setter)
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\bokeh\document\events.py", line 244, in handle_event
 event_cls._handle_event(doc, event)
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\bokeh\document\events.py", line 279, in _handle_event
 cb(event.msg_data)
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\bokeh\document\callbacks.py", line 400, in trigger_event
 model._trigger_event(event)
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\bokeh\util\callback_manager.py", line 111, in _trigger_event
 self.document.callbacks.notify_event(cast(Model, self), event, invoke)
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\bokeh\document\callbacks.py", line 262, in notify_event
 invoke_with_curdoc(doc, callback_invoker)
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\bokeh\document\callbacks.py", line 453, in invoke_with_curdoc
 return f()
 ^^^
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\bokeh\util\callback_manager.py", line 107, in invoke
 cast(EventCallbackWithEvent, callback)(event)
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\panel\reactive.py", line 575, in _comm_event
 state._handle_exception(e)
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\panel\io\state.py", line 496, in _handle_exception
 raise exception
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\panel\reactive.py", line 573, in _comm_event
 self._process_bokeh_event(doc, event)
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\panel\reactive.py", line 510, in _process_bokeh_event
 self._process_event(event)
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\panel\widgets\button.py", line 241, in _process_event
 self.clicks += 1
 ^^^^^^^^^^^
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\param\parameterized.py", line 528, in _f
 instance_param.__set__(obj, val)
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\param\parameterized.py", line 530, in _f
 return f(self, obj, val)
 ^^^^^^^^^^^^^^^^^
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\param\parameters.py", line 543, in __set__
 super().__set__(obj,val)
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\param\parameterized.py", line 530, in _f
 return f(self, obj, val)
 ^^^^^^^^^^^^^^^^^
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\param\parameterized.py", line 1553, in __set__
 obj.param._call_watcher(watcher, event)
 File "C:\Users\E-Store\anaconda3\envs\qutip-env-2\Lib\site-packages\param\parameterized.py", line 2526, in _call_watcher
 self_._execute_watcher(watcher, (event,))
 File "C:\Users\E-Store\anaconda3\envs\qutip

Row
    [0] Column(width=350)
        [0] Markdown(str)
        [1] Divider()
        [2] Markdown(str)
        [3] FloatSlider(end=0.0006, format='0.5e', name='delta_C', step=6e-07, value=0.00011608757555650906)
        [4] FloatSlider(end=100, name='GammaL0', step=0.5, value=50)
        [5] FloatSlider(end=24, name='GammaR0', value=12)
        [6] FloatSlider(end=16, name='Gamma_eg0', value=0.8)
        [7] FloatSlider(end=72, name='Gamma_phi0', value=3.6)
        [8] Divider()
        [9] Markdown(str)
        [10] IntSlider(end=2000, height=35, name='N_steps_period', sizing_mode='stretch_width', start=100, value=1000)
        [11] IntSlider(end=20, height=35, name='N_periods', sizing_mode='stretch_width', start=1, value=10)
        [12] IntSlider(end=10, height=35, name='N_periods_avg', sizing_mode='stretch_width', start=1, value=1)
        [13] Divider()
        [14] Button(button_type='success', name='🔄 Regenerate Data', width=180)
        [15] Markdown(str, sizing_mode='fixed', width=300)
        [16] Markdown(str, sizing_mode='fixed', width=300)
    [1] Column
        [0] Row
            [0] Column
                [0] Markdown(str)
                [1] RadioButtonGroup(button_type='primary', name='Display Level', options=['00', '01', '10', ...], value='dC')
            [1] Column
                [0] FloatSlider(end=10000, name='Color Min', start=-10000, step=10, value=-2000)
                [1] FloatSlider(end=10000, name='Color Max', start=-10000, step=10, value=1000)
        [1] ParamFunction(function, _pane=HoloViews, defer_load=False)